In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_sales
from src.models import SeasonalDateRegressor, SklearnRegressorConfig
from src.pipelines.train import split_by_time
from src.evaluation import regression_metrics
from src.features import add_time_features, add_lag_features, add_rolling_features
from src.pipelines import (
    build_revenue_ratio_targets,
    fit_models_full,
    predict_submission_from_models,
    train_validate_models,
)

data_root = repo_root / "data" / "datathon-2026-round-1"

In [51]:
df = load_sales(data_root=data_root, parse_dates=True)
train_df, valid_df = split_by_time(df, date_col="Date", valid_fraction=0.2)
model = SeasonalDateRegressor()

X_train = train_df[["Date"]]
y_train = train_df["Revenue"]

model.fit(X_train, y_train)

y_pred = model.predict(valid_df[["Date"]])
y_true = valid_df["Revenue"]
metrics = regression_metrics(y_true, y_pred)
metrics

{'mae': 1644555.194360604,
 'rmse': 1965403.7315175715,
 'mape': 69.72676924136569,
 'smape': 47.24199506693533,
 'r2': -0.386503302378604}

In [ ]:
# Feature table
df = load_sales(data_root=data_root, parse_dates=True)
df = add_time_features(df, date_col="Date")
df = add_lag_features(df, target_col="Revenue", lags=[1, 7, 14], date_col="Date")
df = add_rolling_features(df, target_col="Revenue", windows=[7, 14], date_col="Date")
df = df.dropna().reset_index(drop=True)

feature_cols = [col for col in df.columns if col not in ["Date", "Revenue", "COGS"]]
targets = build_revenue_ratio_targets(df, revenue_col="Revenue", cogs_col="COGS", eps=1e-6)
model_config = SklearnRegressorConfig(model_type="random_forest")

# 80/20 validation
validation_result = train_validate_models(
    df=df,
    feature_cols=feature_cols,
    targets=targets,
    model_config=model_config,
    date_col="Date",
    valid_fraction=0.2,
)

print("Revenue metrics:", validation_result["metrics"]["revenue"])
print(
    "Reconstructed COGS metrics:",
    regression_metrics(
        validation_result["valid_df"]["COGS"],
        validation_result["predictions"]["revenue"]
        / validation_result["predictions"]["ratio"].clip(lower=1e-6),
    ),
)

# Full-data retrain for submission forecasting
models = fit_models_full(
    df=df,
    feature_cols=feature_cols,
    targets=targets,
    model_config=model_config,
    use_numpy=True,
)

Revenue metrics: {'mae': 560298.1185008507, 'rmse': 808744.6214439586, 'mape': 23.395869216465027, 'smape': 20.314012107137703, 'r2': 0.7652289065545682}
Reconstructed COGS metrics: {'mae': 513989.96614526986, 'rmse': 752954.0034694931, 'mape': 22.837494301836287, 'smape': 20.733472281211014, 'r2': 0.7301287102355446}


In [ ]:
# Submission prediction from full-data models
submission = predict_submission_from_models(
    models=models,
    feature_cols=feature_cols,
    start_date="2023-01-01",
    end_date="2024-07-01",
    revenue_model_name="revenue",
    ratio_model_name="ratio",
    lags=(1, 7, 14),
    windows=(7, 14),
    eps=1e-6,
)

submission.head(), submission.tail()

d:\MyML\datathon2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
d:\MyML\datathon2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
d:\MyML\datathon2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
d:\MyML\datathon2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
d:\MyML\datathon2026\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warni

(         Date       Revenue          COGS
 0  2023-01-01  9.120988e+05  8.887809e+05
 1  2023-01-02  1.254100e+06  1.086299e+06
 2  2023-01-03  1.430334e+06  1.204956e+06
 3  2023-01-04  1.386725e+06  1.162492e+06
 4  2023-01-05  1.451797e+06  1.210270e+06,
            Date       Revenue          COGS
 543  2024-06-27  7.993873e+06  8.019908e+06
 544  2024-06-28  7.795836e+06  7.794025e+06
 545  2024-06-29  7.652199e+06  7.648957e+06
 546  2024-06-30  7.998704e+06  7.983701e+06
 547  2024-07-01  7.580417e+06  7.570653e+06)

In [ ]:
import os
os.makedirs(repo_root / "submissions", exist_ok=True)
submission.to_csv(repo_root / "submissions" / "baseline_submission.csv", index=False)